# Playground Series S6E7 — Predicting Student Health Risk
## Score: .94997


In [26]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
DATA_DIR = Path("playground-series-s6e7")
TARGET = "health_condition"
ID_COL = "id"

rng = np.random.default_rng(SEED)
print("LightGBM", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

LightGBM 4.6.0 | pandas 3.0.3 | numpy 2.4.6


In [27]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape, "| test:", test.shape)

CAT_COLS = ["diet_type", "stress_level", "sleep_quality",
            "physical_activity_level", "smoking_alcohol", "gender"]
NUM_COLS = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
            "step_count", "exercise_duration", "water_intake"]

CLASSES = ["at-risk", "unhealthy", "fit"]
class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}
y = train[TARGET].map(class_to_int).to_numpy()

print("\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True).round(4))

train: (690088, 15) | test: (295753, 14)

Target distribution:
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


## Feature preparation

In [28]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()

    X["n_missing"] = df[NUM_COLS + CAT_COLS].isna().sum(axis=1)

    X["steps_per_cal"] = X["step_count"] / X["calorie_expenditure"]
    X["cal_per_min_exercise"] = X["calorie_expenditure"] / (X["exercise_duration"] + 1e-3)
    X["steps_per_min_exercise"] = X["step_count"] / (X["exercise_duration"] + 1e-3)

    for c in CAT_COLS:
        X[c] = X[c].fillna("__NA__").astype("category")

    return X


FEATURES = NUM_COLS + CAT_COLS + [
    "n_missing", "steps_per_cal", "cal_per_min_exercise", "steps_per_min_exercise",
]

X = build_features(train)[FEATURES].copy()
X_test = build_features(test)[FEATURES].copy()

for c in CAT_COLS:
    levels = X[c].cat.categories.union(X_test[c].cat.categories)
    X[c] = X[c].cat.set_categories(levels)
    X_test[c] = X_test[c].cat.set_categories(levels)

_alln = pd.concat([train[NUM_COLS], test[NUM_COLS]], ignore_index=True)
mu = _alln.mean()
sd = _alln.std() + 1e-6

DIST_FEATS = []
ztr, zte = {}, {}
for n in NUM_COLS:
    ranks = _alln[n].rank(pct=True)
    X[f"{n}_pct"] = ranks.iloc[:len(train)].to_numpy()
    X_test[f"{n}_pct"] = ranks.iloc[len(train):].to_numpy()
    ztr[n] = (train[n].to_numpy() - mu[n]) / sd[n]
    zte[n] = (test[n].to_numpy() - mu[n]) / sd[n]
    X[f"{n}_z"] = ztr[n]
    X_test[f"{n}_z"] = zte[n]
    DIST_FEATS += [f"{n}_pct", f"{n}_z"]

for D, zd in [(X, ztr), (X_test, zte)]:
    Z = np.vstack([zd[n] for n in NUM_COLS]).T
    D["z_absmean"] = np.nanmean(np.abs(Z), axis=1)
    D["z_max"] = np.nanmax(Z, axis=1)
    D["z_min"] = np.nanmin(Z, axis=1)
    D["z_range"] = D["z_max"] - D["z_min"]
    D["z_nextreme"] = np.nansum(np.abs(Z) > 2, axis=1)
DIST_FEATS += ["z_absmean", "z_max", "z_min", "z_range", "z_nextreme"]

FEATURES = FEATURES + DIST_FEATS
print("Features:", len(FEATURES), "| distributional:", len(DIST_FEATS))
print(X.dtypes)

Features: 36 | distributional: 19
sleep_duration              float64
heart_rate                  float64
bmi                         float64
calorie_expenditure         float64
step_count                  float64
exercise_duration           float64
water_intake                float64
diet_type                  category
stress_level               category
sleep_quality              category
physical_activity_level    category
smoking_alcohol            category
gender                     category
n_missing                     int64
steps_per_cal               float64
cal_per_min_exercise        float64
steps_per_min_exercise      float64
sleep_duration_pct          float64
sleep_duration_z            float64
heart_rate_pct              float64
heart_rate_z                float64
bmi_pct                     float64
bmi_z                       float64
calorie_expenditure_pct     float64
calorie_expenditure_z       float64
step_count_pct              float64
step_count_z                fl

## Cross-validated training

In [29]:
lgb_params = dict(
    objective="multiclass",
    num_class=len(CLASSES),
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

PL_ROUNDS = 2
PL_THRESH = {0: 0.97, 1: 0.85, 2: 0.85}
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

CW_CANDIDATES = [
    "balanced",
    {0: 1.0, 1: 3.0, 2: 4.0},
    {0: 1.0, 1: 6.0, 2: 8.0},
    {0: 1.0, 1: 10.0, 2: 15.0},
    {0: 1.0, 1: 15.0, 2: 20.0},
]


def run_cv(extra_X=None, extra_y=None, class_weight=None):
    params = {**lgb_params, "class_weight": class_weight if class_weight is not None else "balanced"}
    oof = np.zeros((len(X), len(CLASSES)))
    tst = np.zeros((len(X_test), len(CLASSES)))
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        if extra_X is None:
            Xtr, ytr = X.iloc[tr_idx], y[tr_idx]
        else:
            Xtr = pd.concat([X.iloc[tr_idx], extra_X], ignore_index=True)
            ytr = np.r_[y[tr_idx], extra_y]
        model = lgb.LGBMClassifier(**params)
        model.fit(
            Xtr, ytr,
            eval_set=[(X.iloc[va_idx], y[va_idx])],
            eval_metric="multi_logloss",
            categorical_feature=CAT_COLS,
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
        )
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])
        tst += model.predict_proba(X_test) / N_SPLITS
    return oof, tst


def run_pipeline(class_weight, oof0=None, tst0=None):
    if oof0 is None:
        oof, tst = run_cv(class_weight=class_weight)
    else:
        oof, tst = oof0, tst0
    for r in range(1, PL_ROUNDS + 1):
        pred = tst.argmax(1)
        conf = tst.max(1)
        keep = np.zeros(len(X_test), dtype=bool)
        for c in range(len(CLASSES)):
            keep |= (pred == c) & (conf >= PL_THRESH[c])
        X_pl = X_test.iloc[np.flatnonzero(keep)].reset_index(drop=True)
        oof, tst = run_cv(X_pl, pred[keep], class_weight=class_weight)
    return oof, tst


def rank_norm(P):
    R = np.empty_like(P)
    for c in range(P.shape[1]):
        R[:, c] = pd.Series(P[:, c]).rank(pct=True).to_numpy()
    return R


r0 = []
for cw in CW_CANDIDATES:
    oof, tst = run_cv(class_weight=cw)
    bal = balanced_accuracy_score(y, oof.argmax(1))
    print(f"round0 sweep cw={cw} | OOF argmax bal-acc={bal:.5f}")
    r0.append((bal, cw, oof, tst))

r0.sort(key=lambda t: -t[0])
top2 = r0[:2]
print(f"\ntop-2 for PL: {[t[1] for t in top2]}")

pipes = []
for bal0, cw, oof0, tst0 in top2:
    oof, tst = run_pipeline(cw, oof0, tst0)
    bal = balanced_accuracy_score(y, oof.argmax(1))
    print(f"full-PL cw={cw} | OOF argmax bal-acc={bal:.5f}")
    pipes.append((oof, tst))

oof_proba = 0.5 * rank_norm(pipes[0][0]) + 0.5 * rank_norm(pipes[1][0])
test_proba = 0.5 * rank_norm(pipes[0][1]) + 0.5 * rank_norm(pipes[1][1])
argmax_oof = oof_proba.argmax(axis=1)
print(f"blend OOF argmax bal-acc: {balanced_accuracy_score(y, argmax_oof):.5f}")


Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.119746
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.120396
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.123989
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.119371
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.122455
round0 sweep cw=balanced | OOF argmax bal-acc=0.94326
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.0953182
Training until validation scores don't 

## Tune the decision rule for balanced accuracy

In [30]:
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression


def weighted_predict(proba: np.ndarray, w: np.ndarray) -> np.ndarray:
    return (proba * w).argmax(axis=1)


def _neg_bal(w2, proba, y_true):
    w = np.array([1.0, w2[0], w2[1]])
    return -balanced_accuracy_score(y_true, weighted_predict(proba, w))


def optimize_weights(proba, y_true, grid=None):
    grid = np.geomspace(0.3, 30, 30) if grid is None else grid
    best_w = np.array([1.0, 1.0, 1.0])
    best = -_neg_bal([1.0, 1.0], proba, y_true)
    for wu in grid:
        for wf in grid:
            s = -_neg_bal([wu, wf], proba, y_true)
            if s > best:
                best, best_w = s, np.array([1.0, wu, wf])
    res = minimize(_neg_bal, x0=best_w[1:], args=(proba, y_true),
                   method="Nelder-Mead", options=dict(xatol=1e-3, fatol=1e-7, maxiter=1000))
    if -res.fun > best:
        best, best_w = -res.fun, np.array([1.0, res.x[0], res.x[1]])
    return best_w, best


def calibrate(oof_p, test_p, y_true):
    oc, tc = np.zeros_like(oof_p), np.zeros_like(test_p)
    for c in range(oof_p.shape[1]):
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(oof_p[:, c], (y_true == c).astype(float))
        oc[:, c] = iso.transform(oof_p[:, c])
        tc[:, c] = iso.transform(test_p[:, c])
    oc /= oc.sum(axis=1, keepdims=True)
    tc /= tc.sum(axis=1, keepdims=True)
    return oc, tc


w_raw, s_raw = optimize_weights(oof_proba, y)
oof_cal, test_cal = calibrate(oof_proba, test_proba, y)
w_cal, s_cal = optimize_weights(oof_cal, y)

print(f"argmax (no tuning): {balanced_accuracy_score(y, argmax_oof):.5f}")
print(f"raw blend        : {s_raw:.5f}  w={w_raw.round(3)}")
print(f"calibrated blend : {s_cal:.5f}  w={w_cal.round(3)}")

if s_cal >= s_raw:
    oof_final, test_final, best_w, best_score = oof_cal, test_cal, w_cal, s_cal
    print("-> using CALIBRATED probabilities")
else:
    oof_final, test_final, best_w, best_score = oof_proba, test_proba, w_raw, s_raw
    print("-> using RAW probabilities")

tuned_oof = weighted_predict(oof_final, best_w)
print(f"\nFinal OOF balanced accuracy (tuned): {best_score:.5f}")
print("\nConfusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:")
print(confusion_matrix(y, tuned_oof))
print("\n", classification_report(y, tuned_oof, target_names=CLASSES, digits=4))

argmax (no tuning): 0.79346
raw blend        : 0.94974  w=[1.    0.215 0.207]
calibrated blend : 0.94980  w=[ 1.     8.422 14.239]
-> using CALIBRATED probabilities

Final OOF balanced accuracy (tuned): 0.94980

Confusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:
[[555535  23247  13779]
 [  1942  55547    235]
 [  1802    204  37797]]

               precision    recall  f1-score   support

     at-risk     0.9933    0.9375    0.9646    592561
   unhealthy     0.7031    0.9623    0.8126     57724
         fit     0.7295    0.9496    0.8251     39803

    accuracy                         0.9403    690088
   macro avg     0.8087    0.9498    0.8674    690088
weighted avg     0.9538    0.9403    0.9438    690088



## Predict test set & write submission

In [31]:
def load_pred(path):
    s = pd.read_csv(path).set_index(ID_COL).reindex(test[ID_COL]).reset_index(drop=True)
    return s[TARGET].map(class_to_int).to_numpy()


pred_model = weighted_predict(test_final, best_w)
pred_94997 = load_pred("score94997.csv")
pred_94991 = load_pred("score94991.csv")

votes = np.stack([pred_model, pred_94997, pred_94991], axis=1)
test_pred_int = pred_model.copy()
for i in range(len(test_pred_int)):
    u, c = np.unique(votes[i], return_counts=True)
    if c.max() >= 2:
        test_pred_int[i] = u[c.argmax()]

test_pred_lbl = pd.Series(test_pred_int).map(int_to_class)

hist_agree = (pred_94997 == pred_94991)
flip = test_pred_int != pred_model
print(
    f"D three-way vote | model!=94997={(pred_model != pred_94997).sum()} | "
    f"model!=94991={(pred_model != pred_94991).sum()} | "
    f"94997==94991={hist_agree.sum()} | "
    f"hist_agree_vs_model={(hist_agree & (pred_94997 != pred_model)).sum()} | "
    f"flips={flip.sum()}"
)

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred_lbl})
submission.to_csv("submission.csv", index=False)

print("Wrote submission.csv:", submission.shape)
print("\nPredicted label distribution:")
print(submission[TARGET].value_counts(normalize=True).round(4))
print("\nTrain label distribution (for reference):")
print(train[TARGET].value_counts(normalize=True).round(4))
submission.head()


D three-way vote | model!=94997=0 | model!=94991=679 | 94997==94991=295074 | hist_agree_vs_model=0 | flips=0
Wrote submission.csv: (295753, 2)

Predicted label distribution:
health_condition
at-risk      0.8106
unhealthy    0.1143
fit          0.0751
Name: proportion, dtype: float64

Train label distribution (for reference):
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## Diagnostics — is there headroom?

In [32]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


def encode(df):
    e = df.copy()
    for c in CAT_COLS:
        e[c] = e[c].cat.codes
    return e.fillna(-999)


Xa = pd.concat([encode(X), encode(X_test)], ignore_index=True)
ya = np.r_[np.zeros(len(X)), np.ones(len(X_test))]
Xa_tr, Xa_va, ya_tr, ya_va = train_test_split(Xa, ya, test_size=0.3, stratify=ya, random_state=SEED)
adv = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                         random_state=SEED, n_jobs=-1, verbose=-1)
adv.fit(Xa_tr, ya_tr)
adv_auc = roc_auc_score(ya_va, adv.predict_proba(Xa_va)[:, 1])
print(f"[1] Adversarial AUC (0.50 = no train/test shift): {adv_auc:.4f}")

Xe = encode(X)
cat_idx = [Xe.columns.get_loc(c) for c in CAT_COLS]
mi = mutual_info_classif(Xe, y, discrete_features=cat_idx, random_state=SEED)
print("\n[2] Mutual information with target (higher = more signal):")
print(pd.Series(mi, index=Xe.columns).sort_values(ascending=False).round(4).to_string())

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
probe = lgb.LGBMClassifier(objective="multiclass", num_class=len(CLASSES),
    n_estimators=3000, learning_rate=0.05, num_leaves=255, min_child_samples=50,
    class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)
probe.fit(Xtr, ytr, categorical_feature=CAT_COLS)
print(f"\n[3] High-capacity probe val balanced-acc (argmax): "
      f"{balanced_accuracy_score(yva, probe.predict(Xva)):.5f}")

Xte = encode(X)
Xtr, Xva, ytr, yva = train_test_split(Xte, y, test_size=0.2, stratify=y, random_state=SEED)
print("\n[4] Decision-tree depth sweep (a big jump = rule-based target):")
for d in [3, 5, 8, 12, None]:
    t = DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=SEED)
    t.fit(Xtr, ytr)
    print(f"    depth={str(d):>4}: val bal-acc={balanced_accuracy_score(yva, t.predict(Xva)):.5f}")

[1] Adversarial AUC (0.50 = no train/test shift): 0.6527

[2] Mutual information with target (higher = more signal):
sleep_duration_pct         0.1581
sleep_duration             0.1580
sleep_duration_z           0.1580
stress_level               0.1426
z_min                      0.1044
physical_activity_level    0.0560
z_max                      0.0448
bmi                        0.0325
bmi_pct                    0.0323
bmi_z                      0.0316
n_missing                  0.0270
exercise_duration          0.0237
exercise_duration_z        0.0234
exercise_duration_pct      0.0234
step_count_z               0.0226
step_count_pct             0.0223
step_count                 0.0221
steps_per_cal              0.0188
cal_per_min_exercise       0.0146
sleep_quality              0.0142
steps_per_min_exercise     0.0123
water_intake_pct           0.0114
water_intake_z             0.0112
water_intake               0.0108
z_range                    0.0104
z_nextreme                 0.0086